# Lesson 3.2 — Authentication and Authorization Gap Analysis

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Separate authentication (who) from authorization (what they may do)
- Run a gap analysis over an integration to find security holes
- Recommend fixes that enforce per-request scope

**You will produce:** An `authz_gaps` checklist flagging missing identity and scope controls.

## Concepts

**Authentication (authN)** verifies *who* is making a request (API keys, OAuth, SSO). **Authorization (authZ)** decides *what* that identity may do (scopes, roles, row-level access). LLM integrations fail when they authenticate the *service* but then let the model act with the service's full privileges on behalf of *any* end user — the **confused deputy** problem.

A gap analysis walks the request path and asks, at each hop: Is the caller authenticated? Is the action authorized *for this end user*? Is sensitive data scoped to them? Are tool calls executed with the end user's permissions, not the agent's superuser creds? The architect's deliverable is a checklist that turns these questions into pass/fail controls.

> **Reference:** [Anthropic — Security & best practices](https://docs.anthropic.com/en/docs/overview)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Model the request path and its controls

We represent each hop with the authN/authZ controls it should enforce, then look for missing ones.

In [ ]:
request_path = [
  {"hop": "client -> API",        "authn": True,  "authz": True,  "scoped_to_user": True},
  {"hop": "API -> agent",          "authn": True,  "authz": False, "scoped_to_user": False},
  {"hop": "agent -> tool (DB)",     "authn": True,  "authz": False, "scoped_to_user": False},
  {"hop": "agent -> vector store",  "authn": True,  "authz": True,  "scoped_to_user": False},
]
for h in request_path: print(h)

## Code-Along: Step 2 — Run the gap analysis

We flag any hop that authenticates but fails to authorize or scope to the end user — the classic confused-deputy gap where the agent over-reaches.

In [ ]:
def authz_gap_analysis(path):
    gaps = []
    for h in path:
        if h["authn"] and not h["authz"]:
            gaps.append(f"{h['hop']}: authenticated but NOT authorized")
        if not h["scoped_to_user"]:
            gaps.append(f"{h['hop']}: action not scoped to end user (confused-deputy risk)")
    return gaps or ["PASS: every hop authorized and user-scoped"]

for g in authz_gap_analysis(request_path):
    print("-", g)

## Exercise

Close the gaps: update `request_path` so the agent passes the **end user's** scoped token to the DB and vector store (row-level/user-scoped retrieval), then re-run the analysis to a PASS. Add a one-line note on why agent-level superuser creds are dangerous for RAG.

In [ ]:
# TODO: fix authz/scoped_to_user on the failing hops
# TODO: re-run authz_gap_analysis and confirm PASS
# TODO: note why unscoped retrieval can leak another user's documents

## Key Takeaways

- AuthN is identity; authZ is permission — integrations need both at every hop.
- The confused-deputy problem: the agent acts with its own broad creds, not the user's scope.
- RAG retrieval must be scoped to the end user or it leaks data across tenants.
- A gap-analysis checklist turns security review into repeatable pass/fail controls.

---

**Next:** [Lesson 3.3 — Accuracy vs Latency Trade-offs](./lesson-3.3-accuracy-latency-tradeoffs.ipynb)